In [2]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
# Use the ezy_seq bundled with THIS repo (EzySeq_Library/mypythonlibrary/src),
# NOT any pip-installed copy. Walk up from the CWD to find the repo's library and
# put it first on sys.path so `import ezy_seq` resolves to it.
for _d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    _ezy_src = _d / "EzySeq_Library" / "mypythonlibrary" / "src"
    if _ezy_src.is_dir():
        sys.path.insert(0, str(_ezy_src))
        break

import ezy_seq as ezy
print("ezy_seq from:", ezy.__file__)
#import ezy_seq.load 
import scanpy as sc
import pandas as pd
import os
from pathlib import Path  # For file path operations
import matplotlib.pyplot as plt
import napari
import glob
import numpy as np
import pathlib
from matplotlib.path import Path as MPath  # For polygon operations (matplotlib Path)
import anndata as ad
from scipy import sparse
import random
random.seed(0)


c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


ezy_seq from: c:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\EzySeq_Library\mypythonlibrary\src\ezy_seq\__init__.py


c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [3]:
# ── Load pipeline paths from config.yaml (per framework) ─────────────────────
# Single source of truth for every file location (config.yaml lives at the repo
# root). Edit config.yaml, not the path lines in the cells below.
#
# The pipeline runs over two frameworks (datasets/contrasts): "genotype" and
# "fmt". Pick one here — each has its own inputs and its own LABELLED outputs, so
# preprocessing the two datasets never overwrites the other's files.
import os
import yaml
from pathlib import Path

def _find_config(name="config.yaml"):
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found in {Path.cwd()} or its parents")

def _select_framework(cfg, name=None):
    if "frameworks" not in cfg:            # flat/legacy config
        return cfg
    name = name or os.environ.get("PIPELINE_FRAMEWORK") or cfg.get("active_framework")
    if name not in cfg["frameworks"]:
        raise KeyError(f"Framework {name!r} not in config. Available: {list(cfg['frameworks'])}")
    print(f"Framework: {name}")
    return cfg["frameworks"][name]

def _abs_paths(section, base):
    # config.yaml paths are relative TO THE CONFIG'S DIRECTORY (repo root), not to
    # the kernel's working dir (this notebook lives in Pre_processing/). Make each
    # relative path absolute against `base`. Keys ending in "_col" are column
    # names, not paths, so leave those (and any absolute value) untouched.
    out = {}
    for k, v in section.items():
        if isinstance(v, str) and not k.endswith("_col") and not Path(v).is_absolute():
            out[k] = str((base / v).resolve())
        else:
            out[k] = v
    return out

_CONFIG_PATH = _find_config().resolve()
_CONFIG_DIR = _CONFIG_PATH.parent          # repo root — paths resolve against this
cfg = yaml.safe_load(open(_CONFIG_PATH))

# Which framework/dataset to run. Override via env PIPELINE_FRAMEWORK, or set
# FRAMEWORK explicitly here ("genotype" or "fmt"); None -> config active_framework.
FRAMEWORK = "fmt"
_fw = _select_framework(cfg, FRAMEWORK)
INPUTS  = _abs_paths(_fw["inputs"],  _CONFIG_DIR)
OUTPUTS = _abs_paths(_fw["outputs"], _CONFIG_DIR)
COMP    = _fw["composition"]

print(f"Config:      {_CONFIG_PATH}")
print(f"sample_info: {INPUTS['sample_info_xlsx']}")
print(f"cell_type_col: {INPUTS['cell_type_col']}")


Framework: fmt
Config:      C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\config.yaml
sample_info: C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\sample_info_FMT.xlsx
cell_type_col: RNA_Rerun_w_CosMx_Profile_Cell.Typing.InSituType.1_1_clusters


Load CosMx Exports directly into AnnData object

In [5]:
r"""First, Load "polygons.csv.gz" files for cellwise labeling in Napari"""
cell_type_col_name = INPUTS["cell_type_col"]  # source cell-type/cluster column (config.yaml); renamed to 'cell_type' downstream
file_path = INPUTS["sample_info_xlsx"]  # samplewise metadata workbook (see config.yaml)
meta_dictionary=ezy.read_dictionary(file_path)
CosMx_Export_Path=INPUTS["cosmx_export_dir"]
adatas=ezy.load.cosmx(CosMx_Export_Path)



################################################################################################################################
def load_polys(fp):
    if not os.path.exists(fp): return []
    df = pd.read_csv(fp)
    axes = ['axis-0','axis-1']
    xcol,ycol = axes[1], axes[0]
    df = df.dropna(subset=[xcol,ycol])
    return [g[[ycol,xcol]].to_numpy() for _,g in df.groupby('index')]

def load_all_polygons(project_folder):
    project_folder_ = pathlib.Path(project_folder)
    polygons = {}
    # Key each polygon table by its slide FOLDER name — the same identity
    # load.cosmx() assigns to obs['slide_ID'] — so keys stay consistent even when
    # the files carry a GEO/GSM (or any other) prefix. Match .csv.gz or plain .csv.
    for pattern in ("*-polygons.csv.gz", "*-polygons.csv"):
        for fp in project_folder_.rglob(pattern):
            slide_name = fp.parent.name
            if slide_name not in polygons:
                polygons[slide_name] = pd.read_csv(fp)
    return polygons

project_folder = CosMx_Export_Path
polygons_dict = load_all_polygons(project_folder)

# Dynamic: follow whatever slides load.cosmx() actually loaded (in load order),
# instead of a hard-coded slide list. slide_ID == the raw export's folder name,
# which now matches the polygons_dict keys above.
layer_folders = [a.obs['slide_ID'].unique()[0] for a in adatas]
missing = [f for f in layer_folders if f not in polygons_dict]
if missing:
    print(f"[warn] no *-polygons file found for slide(s): {missing}")
polygons = [polygons_dict[f] for f in layer_folders if f in polygons_dict]
################################################################################################################################


Detected 'flatFiles' folder directly in root: C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\raw_data\FMT_Comp\flatFiles
Found 5 dataset(s) to process.
Processing: 118Band1182L
Processing: 120Land323DY
Processing: 217Rand144_2L
Processing: 367Rand215L
Processing: 368R


In [ ]:
r"""Open slides in Napari to generate sample-defining .csvs (only necassary if multiple samples are per slide)."""
import imageio.v2 as imageio
import napari
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Existing napari definitions overlaid on each slide ───────────────────────
# napari_definitions/<framework>/<slide_ID>/ holds BOTH sample-defining and
# region-defining shape exports side by side. Both are drawn here, styled apart:
#   sample polygons -> mostly translucent white
#   region polygons -> one stable colour per region, consistent across slides
# A file is treated as a sample when its stem matches a sample_ID in the metadata
# workbook; anything else is a region, except the stray napari exports below.
SAMPLE_POLY_DIR = pathlib.Path(INPUTS["napari_sample_polygons_dir"])
REGION_POLY_DIR = pathlib.Path(INPUTS["napari_region_polygons_dir"])

# Not shape definitions at all - napari exports of the CosMx layers themselves.
JUNK_STEMS = {"CosMx cells", "CosMx polygons"}

# Fallback, used only if sample_ID cannot be recovered from the workbook.
NON_SAMPLE_STEMS = {"Cortex", "Hippocampus", "Striatum", "Cerebellum", "Olfactory",
                    "Unassigned"}

# RGBA, 0-1. Faces stay faint so the cells underneath remain visible; edges are
# far more opaque so boundaries read clearly.
POLY_FACE_COLOR = (1.0, 1.0, 1.0, 0.35)   # sample fill: translucent white
POLY_EDGE_COLOR = (1.0, 1.0, 1.0, 1.0)    # sample outline: solid white
REGION_FACE_ALPHA = 0.35
REGION_EDGE_ALPHA = 1.0
SAMPLE_EDGE_WIDTH = 60
REGION_EDGE_WIDTH = 40


def _known_sample_ids(meta):
    """sample_ID values from the metadata workbook; empty set if unavailable."""
    try:
        sheets = list(meta.keys())
    except AttributeError:
        return set()
    for sheet in ["sample_metadata", *sheets]:
        df = meta.get(sheet)
        if isinstance(df, pd.DataFrame) and "sample_ID" in df.columns:
            return set(df["sample_ID"].dropna().astype(str))
    return set()


SAMPLE_IDS = _known_sample_ids(meta_dictionary)
if SAMPLE_IDS:
    print(f"{len(SAMPLE_IDS)} sample IDs from metadata: {sorted(SAMPLE_IDS)}")
else:
    print(f"sample_ID not found in metadata; treating {sorted(NON_SAMPLE_STEMS)} as regions")


def classify_csv(p):
    """'sample', 'region', or None for files that are not shape definitions."""
    if p.stem in JUNK_STEMS:
        return None
    if SAMPLE_IDS:
        return "sample" if p.stem in SAMPLE_IDS else "region"
    return "region" if p.stem in NON_SAMPLE_STEMS else "sample"


def slide_csvs(slide_id):
    """Every definition CSV for a slide, across both configured folders."""
    seen, out = set(), []
    for root in (SAMPLE_POLY_DIR, REGION_POLY_DIR):
        d = root / str(slide_id)
        if not d.is_dir():
            continue
        for p in sorted(d.glob("*.csv")):
            rp = p.resolve()
            if rp not in seen:
                seen.add(rp)
                out.append(p)
    return out


# Stable region -> colour map: built once from every slide folder, so a region
# keeps the same colour no matter which slide is being viewed.
_all_slide_dirs = {d for root in (SAMPLE_POLY_DIR, REGION_POLY_DIR)
                   if root.is_dir() for d in root.iterdir() if d.is_dir()}
ALL_REGIONS = sorted({p.stem for d in _all_slide_dirs
                      for p in d.glob("*.csv") if classify_csv(p) == "region"})
_region_cmap = plt.get_cmap("tab10")
REGION_COLORS = {r: _region_cmap(i % 10) for i, r in enumerate(ALL_REGIONS)}
print(f"{len(ALL_REGIONS)} regions found: {ALL_REGIONS}")


def load_shape_csv(fp):
    """napari shapes export -> (list of vertex arrays, list of shape types)."""
    df = pd.read_csv(fp).dropna(subset=["axis-0", "axis-1"])
    verts, kinds = [], []
    for _, g in df.groupby("index", sort=True):
        if "vertex-index" in g.columns:
            g = g.sort_values("vertex-index")
        verts.append(g[["axis-0", "axis-1"]].to_numpy())
        kind = str(g["shape-type"].iloc[0]) if "shape-type" in g.columns else "polygon"
        # napari's rectangle/ellipse primitives require exactly 4 vertices.
        if kind in ("rectangle", "ellipse") and len(g) != 4:
            kind = "polygon"
        kinds.append(kind)
    return verts, kinds


def add_definition_polygons(viewer, slide_id):
    """Add every sample and region definition for this slide as its own layer."""
    csvs = slide_csvs(slide_id)
    if not csvs:
        print(f"   no napari definitions found for slide {slide_id}")
        return

    n = {"sample": 0, "region": 0}
    for fp in csvs:
        kind_of = classify_csv(fp)
        if kind_of is None:
            continue
        verts, kinds = load_shape_csv(fp)
        if not verts:
            print(f"   ! {fp.stem}: no usable shapes, skipped")
            continue

        if kind_of == "sample":
            face, edge, width = POLY_FACE_COLOR, POLY_EDGE_COLOR, SAMPLE_EDGE_WIDTH
        else:
            r, g, b, _ = REGION_COLORS.get(fp.stem, (0.5, 0.5, 0.5, 1.0))
            face = (r, g, b, REGION_FACE_ALPHA)
            edge = (r, g, b, REGION_EDGE_ALPHA)
            width = REGION_EDGE_WIDTH

        viewer.add_shapes(
            verts,
            shape_type=kinds,
            name=fp.stem,                              # layer labelled by its CSV
            edge_color=[edge] * len(verts),
            face_color=[face] * len(verts),
            edge_width=width,
            opacity=1.0,
        )
        n[kind_of] += 1
        print(f"   + [{kind_of}] {fp.stem}: {len(verts)} shape(s)")

    print(f"   -> {n['sample']} sample layer(s), {n['region']} region layer(s)")


for slide in adatas:
    # Create subset for the current slide
    s_df = slide
    slide_id = slide.obs['slide_ID'].iloc[0]

    print(f"Processing slide: {slide_id}")

    # --- PREPARE COLORS ---
    cluster_key=cell_type_col_name

    unique_clusters = s_df.obs[cluster_key].unique()
    cmap = plt.get_cmap('tab20')
    colors_list = [cmap(i) for i in np.linspace(0, 1, len(unique_clusters))]

    cluster_color_dict = dict(zip(unique_clusters, colors_list))
    assigned_colors = s_df.obs[cluster_key].astype(object).map(cluster_color_dict).tolist()

    # --- INITIALIZE VIEWER & ADD POINTS ---
    viewer = napari.Viewer()
    viewer.add_points(
        s_df.obsm['spatial_fov'],
        size=100,
        border_width=0.0,
        face_color=assigned_colors,
        name="CosMx cells"
    )

    # --- OVERLAY THE SAMPLE + REGION DEFINITIONS FOR THIS SLIDE ---
    add_definition_polygons(viewer, slide_id)

napari.run()


In [9]:
r"""Use Napari-generated .csv files to define samples (only necassary if multiple samples are per slide)."""
processed_adatas = []
base=INPUTS["napari_sample_polygons_dir"]
# Zip the anndata objects with their corresponding folder
for adata_full in adatas:
    if adata_full.obs['slide_ID'].unique().size>1:
        print('we have multiple slides in this object')
        continue
    slide_folder=adata_full.obs['slide_ID'].unique()[0]
    print(f"Processing slide folder: {slide_folder}")
    s_adata = adata_full.copy()
    s_adata.obs['sample_ID'] = slide_folder
    

    cell_xy = np.column_stack((s_adata.obsm['spatial_fov'][:, 1], s_adata.obsm['spatial_fov'][:, 0]))
    search_path = os.path.join(base, slide_folder, "*.csv")
    print('##############################################################')
    print(search_path)
    print('##############################################################')

    # Accept both uncompressed (*.csv) and gzipped (*.csv.gz) sample polygons.
    found_csvs = glob.glob(search_path) + glob.glob(search_path + ".gz")
    
    if not found_csvs:
        print(f"  No CSVs found in {slide_folder}")
    
    for fp in found_csvs:
        # Extract filename to use as Sample ID (strip .csv or .csv.gz)
        file_name = os.path.basename(fp)
        sample_code = file_name[:-7] if file_name.endswith(".csv.gz") else os.path.splitext(file_name)[0]
        if not sample_code or not sample_code[0].isdigit():
            print(f"  -> Skipping {sample_code} ")
            continue
        try:
            df = pd.read_csv(fp)   # pandas infers gzip from the .gz extension
        except Exception as e:
            print(f"  Error reading {fp}: {e}")
            continue

        # Detect axis columns
        xcol = 'axis-1' if 'axis-1' in df.columns else None
        ycol = 'axis-0' if 'axis-0' in df.columns else None

        if xcol is None or ycol is None:
            print(f"  Bad cols in {fp}; need axis-0/axis-1")
            continue
        hit_mask = np.zeros(len(cell_xy), dtype=bool)
        
        for _, g in df.groupby('index'):
            verts = np.column_stack((g[xcol].to_numpy(dtype=float), g[ycol].to_numpy(dtype=float)))
            if len(verts) < 3: 
                continue
            p = MPath(verts)
            hit_mask |= p.contains_points(cell_xy)

        if hit_mask.any():
            s_adata.obs.loc[hit_mask, 'sample_ID'] = sample_code
            print(f"  Assigned '{sample_code}': {hit_mask.sum()} cells")
    
    print(f"Finished {slide_folder} — Distribution:\n{s_adata.obs['sample_ID'].value_counts()}\n")
    processed_adatas.append(s_adata)

# Merge all slides into one object
#adata_full_pre = sc.concat(processed_adatas, join='outer', axis=0, merge="first", index_unique='-')


Processing slide folder: 118Band1182L
##############################################################
C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\napari_definitions\fmt\118Band1182L\*.csv
##############################################################
  Assigned '1182L': 67551 cells
  Assigned '118B': 97260 cells
  -> Skipping Cerebellum 
  -> Skipping Cortex 
  -> Skipping Hippocampus 
  -> Skipping Olfactory 
  -> Skipping Striatum 
Finished 118Band1182L — Distribution:
sample_ID
118B     97260
1182L    67551
Name: count, dtype: int64

Processing slide folder: 120Land323DY
##############################################################
C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\napari_definitions\fmt\120Land323DY\*.csv
##############################################################
  Assigned '120L': 95700 cells
  Ass

In [10]:
#combinining

annotate_adatas=ezy.apply_annotation(processed_adatas,meta_dictionary,cell_type_key=cell_type_col_name)
norm_adatas=ezy.filter_and_normalize(annotate_adatas
    ,min_gene_cnt=20
    ,min_t_cnt=100)
adata_full=sc.concat(norm_adatas, join='outer', axis=0, merge="first", index_unique='-')

adata_full.obs.rename(columns={cell_type_col_name: "cell_type"}, inplace=True)

# ── Restrict to the framework's analysis cohort (config.yaml -> composition.cohort_filter) ──
# For the genotype framework this drops every sample with FMT != MCI; for the fmt
# framework cohort_filter is empty ({}) so nothing is removed. Applied here so the
# saved adata_full.h5ad (and the QUINT export) already contain only cohort cells.
for _col, _allowed in (COMP.get("cohort_filter") or {}).items():
    adata_full = adata_full[adata_full.obs[_col].isin(list(_allowed))].copy()


***********************
Processing: 118Band1182L
***********************
118B
    Assigned FMT: Cntrl
1182L
    Assigned FMT: Cntrl
***********************
Processing: 120Land323DY
***********************
120L
    Assigned FMT: Stroke_FMT
323DY
    Assigned FMT: Healthy_FMT
***********************
Processing: 217Rand144_2L
***********************
217R
    Assigned FMT: Healthy_FMT
144_2L
    Assigned FMT: Cntrl
***********************
Processing: 367Rand215L
***********************
367R
    Assigned FMT: Healthy_FMT
215L
    Assigned FMT: Stroke_FMT
***********************
Processing: 368R
***********************
368R
    Assigned FMT: Stroke_FMT


c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\scanpy\preprocessing\_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\scanpy\preprocessing\_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\scanpy\preprocessing\_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\scanpy\preprocessing\_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\scanpy\preprocessing\_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\scanpy\preprocessing\_n

In [14]:

base = INPUTS["napari_region_polygons_dir"]
dfs = []
# IMPORTANT: Order matters! Regions applied later OVERRIDE earlier ones where
# polygons overlap. Files are processed in sorted order, so e.g. 'Prefrontal' (P)
# is applied after 'Cortex' (C) and wins the overlap. Adjust the sort if you need
# a different precedence.

def find_region_csvs(folder):
    """Region polygons = every .csv/.csv.gz in `folder` whose name does NOT start
    with a digit (digit-named files are per-sample polygons, not regions)."""
    names = set()
    for fp in glob.glob(os.path.join(folder, "*.csv")) + glob.glob(os.path.join(folder, "*.csv.gz")):
        fname = os.path.basename(fp)
        stem = fname[:-7] if fname.endswith(".csv.gz") else fname[:-4]
        if not stem or stem[0].isdigit():
            continue          # skip sample polygons (e.g. 118B.csv, 1182L.csv)
        if "CosMx" in stem:
            continue          # skip raw CosMx cell/polygon dumps
        names.add(stem)
    return sorted(names)

# Loop through your sample lists and corresponding folders
for slide_folder in adata_full.obs['slide_ID'].unique():

    # Create the subset for the current slide/batch
    s_df = adata_full[adata_full.obs['slide_ID'] == slide_folder].copy()
    s_df.obs['napari_region'] = 'Unassigned'

    cell_xy = s_df.obsm['spatial_fov'][:, :2]
    slide_path = os.path.join(base, slide_folder)

    # Dynamically discover the region polygon files in this slide's folder.
    categories = find_region_csvs(slide_path)
    print(f"{slide_folder}: regions found -> {categories}")

    for region_name in categories:
        # Accept either an uncompressed (.csv) or a gzipped (.csv.gz) polygon file.
        fp = os.path.join(slide_path, region_name + ".csv")
        if not os.path.exists(fp):
            fp = fp + ".gz"
        if not os.path.exists(fp):
            continue

        df = pd.read_csv(fp)   # pandas infers gzip compression from the .gz extension
        # Detect axis columns: axis-1 is X, axis-0 is Y in Napari
        xcol = 'axis-0' if 'axis-0' in df.columns else None
        ycol = 'axis-1' if 'axis-1' in df.columns else None

        if xcol is None or ycol is None:
            print(f"Skipping {fp}: columns need to be axis-0 and axis-1")
            continue
        if 'index' not in df.columns:
            print(f"Skipping {fp}: no 'index' column")
            continue

        # Group vertices by polygon index and test points
        hit_mask = np.zeros(len(cell_xy), dtype=bool)

        for _, g in df.groupby('index'):
            # Stack as (X, Y) to match cell_xy
            verts = np.column_stack((g[xcol].to_numpy(dtype=float), g[ycol].to_numpy(dtype=float)))
            if len(verts) < 3:  # skip degenerate polygons
                continue
            p = MPath(verts)
            # Update mask: true if point is in ANY of the polygons for this region type
            hit_mask |= p.contains_points(cell_xy)

        # Assign the region name ONLY to the 'napari_region' column
        if hit_mask.any():
            s_df.obs.loc[hit_mask, 'napari_region'] = region_name
            print(f"  -> Assigned '{region_name}': {hit_mask.sum()} cells")

    # Append processed subset to list
    print(f"Finished {slide_folder} — Region distribution:\n{s_df.obs['napari_region'].value_counts()}\n")
    dfs.append(s_df)

# Concatenate back together
adata_full = sc.concat(dfs, join='outer', axis=0, merge="first", index_unique='-')

120Land323DY: regions found -> ['Cerebellum', 'Cortex', 'Hippocampus', 'Olfactory', 'Striatum']
  -> Assigned 'Cerebellum': 32535 cells
  -> Assigned 'Cortex': 48986 cells
  -> Assigned 'Hippocampus': 7468 cells
  -> Assigned 'Olfactory': 12568 cells
  -> Assigned 'Striatum': 12433 cells
Finished 120Land323DY — Region distribution:
napari_region
Unassigned     85011
Cortex         48986
Cerebellum     32535
Olfactory      12568
Striatum       12433
Hippocampus     7468
Name: count, dtype: int64

217Rand144_2L: regions found -> ['Cortex', 'Hippocampus', 'Striatum']
  -> Assigned 'Cortex': 10582 cells
  -> Assigned 'Hippocampus': 3015 cells
  -> Assigned 'Striatum': 10004 cells
Finished 217Rand144_2L — Region distribution:
napari_region
Unassigned     27013
Cortex         10582
Striatum       10004
Hippocampus     3015
Name: count, dtype: int64

367Rand215L: regions found -> ['Cortex', 'Hippocampus', 'Striatum']
  -> Assigned 'Cortex': 23973 cells
  -> Assigned 'Hippocampus': 7907 cells


In [13]:
r"""Validate labeling accuracy"""
import imageio.v2 as imageio
import napari
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Base directory for your project

for sample in adata_full.obs['sample_ID'].unique():
    # Create subset for the current slide
    s_df = adata_full[adata_full.obs['sample_ID']==sample]
    slide_FMT=s_df.obs['FMT'].unique()[0]
    print(f"Processing slide: {sample}")
    
    # --- PREPARE COLORS ---
    cluster_key='napari_region'

    unique_clusters = s_df.obs[cluster_key].unique()
    cmap = plt.get_cmap('tab20') 
    colors_list = [cmap(i) for i in np.linspace(0, 1, len(unique_clusters))]
    
    cluster_color_dict = dict(zip(unique_clusters, colors_list))
    assigned_colors = s_df.obs[cluster_key].astype(object).map(cluster_color_dict).tolist()

    # --- INITIALIZE VIEWER & ADD POINTS ---
    viewer = napari.Viewer()
    viewer.add_points(
        s_df.obsm['spatial_fov'], 
        size=100,
        border_width=0.0,
        face_color=assigned_colors,
        name=slide_FMT
    )
napari.run()

Processing slide: 120L
Processing slide: 323DY
Processing slide: 217R
Processing slide: 367R
Processing slide: 215L
Processing slide: 368R


Re-defining Block

In [14]:
base = r"C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\napari_definitions\fmt"
slide_folders = [f for f in os.listdir(base) if os.path.isdir(os.path.join(base, f))]

for sample in adata_full.obs['sample_ID'].unique():
    # Create subset for the current sample
    s_df = adata_full[adata_full.obs['sample_ID'] == sample]
    slide_FMT = s_df.obs['FMT'].unique()[0]
    print(f"Processing slide: {sample}")

    # --- PREPARE COLORS ---
    cluster_key = 'cell_type'
    unique_clusters = s_df.obs[cluster_key].unique()
    cmap = plt.get_cmap('tab20')
    colors_list = [cmap(i) for i in np.linspace(0, 1, len(unique_clusters))]
    cluster_color_dict = dict(zip(unique_clusters, colors_list))
    assigned_colors = s_df.obs[cluster_key].astype(object).map(cluster_color_dict).tolist()

    # --- INITIALIZE VIEWER & ADD POINTS ---
    viewer = napari.Viewer()
    viewer.add_points(
        s_df.obsm['spatial_fov'],
        size=100,
        border_width=0.0,
        face_color=assigned_colors,
        name=slide_FMT
    )

    # --- ADD NAPARI POLYGONS: folder whose name contains this sample_ID ---
    match = next((f for f in slide_folders if sample in f), None)
    if match is None:
        print(f"  -> No folder matching sample '{sample}' in {base}")
    else:
        folder_path = os.path.join(base, match)
        for fp in glob.glob(os.path.join(folder_path, "*.csv")):
            region_name = os.path.splitext(os.path.basename(fp))[0]
            if "CosMx" in region_name:          # skip the raw CosMx cell/polygon dumps
                continue
            df = pd.read_csv(fp)
            if 'index' not in df.columns:
                continue
            xcol = 'axis-0' if 'axis-0' in df.columns else None
            ycol = 'axis-1' if 'axis-1' in df.columns else None
            if xcol is None or ycol is None:
                continue
            polygons_list = []
            for _, g in df.groupby('index'):
                verts = np.column_stack((g[xcol].to_numpy(dtype=float),
                                         g[ycol].to_numpy(dtype=float)))
                if len(verts) < 3:
                    continue
                polygons_list.append(verts)
            if polygons_list:
                viewer.add_shapes(
                    polygons_list,
                    shape_type='polygon',
                    edge_width=20,
                    edge_color='white',
                    face_color=[0.8, 0.8, 0.8, 0.3],   # low alpha so points show through
                    name=f"{sample}_{region_name}"
                )
                print(f"  -> Added '{region_name}': {len(polygons_list)} polygon(s)")

napari.run()

Processing slide: 120L
  -> Added '120L': 1 polygon(s)
  -> Added '323DY': 1 polygon(s)
  -> Added 'Cerebellum': 3 polygon(s)
  -> Added 'Cortex': 5 polygon(s)
  -> Added 'Hippocampus': 2 polygon(s)
  -> Added 'Olfactory': 1 polygon(s)
  -> Added 'Striatum': 1 polygon(s)
Processing slide: 323DY
  -> Added '120L': 1 polygon(s)
  -> Added '323DY': 1 polygon(s)
  -> Added 'Cerebellum': 3 polygon(s)
  -> Added 'Cortex': 5 polygon(s)
  -> Added 'Hippocampus': 2 polygon(s)
  -> Added 'Olfactory': 1 polygon(s)
  -> Added 'Striatum': 1 polygon(s)
Processing slide: 217R
  -> Added '144_2L': 1 polygon(s)
  -> Added '217R': 1 polygon(s)
  -> Added 'Cortex': 6 polygon(s)
  -> Added 'Hippocampus': 2 polygon(s)
  -> Added 'Striatum': 4 polygon(s)
Processing slide: 367R
  -> Added '215L': 1 polygon(s)
  -> Added '367R': 1 polygon(s)
  -> Added 'Cortex': 3 polygon(s)
  -> Added 'Hippocampus': 3 polygon(s)
  -> Added 'Striatum': 2 polygon(s)
Processing slide: 215L
  -> Added '215L': 1 polygon(s)
  -> A

Export For QUINT labeling

In [38]:


def export_anndata_for_seurat(
    adata,
    output_dir,
    *,
    counts_layer_candidates=("counts", "raw"),
    x_filename="normalized_counts.csv",
    counts_filename="raw_counts.csv",
    var_filename="features_counts.csv",
    obs_filename="cell_metadata.csv",
    coords_key="spatial_fov",
    coords_filename="coords_xy.csv",
    pca_key="X_pca",
    pca_filename="pca.csv",
    umap_key="X_umap",
    umap_filename="umap.csv",
    export_other_obsm=True,
    other_obsm_exclude=("spatial_fov", "X_pca", "X_umap"),
    verbose=True,
):
    """
    Export an AnnData object to CSVs for reconstruction in R/Seurat.

    Writes normalized_counts.csv, raw_counts.csv, features_counts.csv,
    cell_metadata.csv, coords_xy.csv, pca.csv, umap.csv, and any other
    obsm matrices as <key>.csv.  Returns a dict of written file paths.
    """
    import os
    import numpy as np
    import pandas as pd
    from scipy import sparse

    os.makedirs(output_dir, exist_ok=True)
    written = {}

    def _dense(mat):
        if sparse.issparse(mat): return mat.toarray()
        if hasattr(mat, "toarray"): return mat.toarray()
        return np.asarray(mat)

    # Raw counts
    layer = next((l for l in counts_layer_candidates if l in getattr(adata, "layers", {})), None)
    counts = adata.layers[layer] if layer else adata.X
    if layer is None and verbose:
        print("Warning: no counts layer found; using X for raw counts")
    fp = os.path.join(output_dir, counts_filename)
    pd.DataFrame(_dense(counts), index=adata.obs_names, columns=adata.var_names).to_csv(fp)
    written["counts"] = fp

    # Normalized (X)
    fp = os.path.join(output_dir, x_filename)
    pd.DataFrame(_dense(adata.X), index=adata.obs_names, columns=adata.var_names).to_csv(fp)
    written["X"] = fp

    # Metadata
    for name, fname in [(adata.var, var_filename), (adata.obs, obs_filename)]:
        fp = os.path.join(output_dir, fname)
        name.to_csv(fp)
        written[fname] = fp

    # Spatial coords
    if coords_key in getattr(adata, "obsm", {}):
        fp = os.path.join(output_dir, coords_filename)
        coords = np.asarray(adata.obsm[coords_key])[:, :2]
        pd.DataFrame(coords, index=adata.obs_names, columns=["x", "y"]).to_csv(fp)
        written["coords"] = fp

    # PCA / UMAP
    for key, fname, col_fmt in [(pca_key, pca_filename, "PC{}"), (umap_key, umap_filename, "UMAP_{}")]:
        if key in getattr(adata, "obsm", {}):
            arr = np.asarray(adata.obsm[key])
            cols = [col_fmt.format(i + 1) for i in range(arr.shape[1])]
            fp = os.path.join(output_dir, fname)
            pd.DataFrame(arr, index=adata.obs_names, columns=cols).to_csv(fp)
            written[key] = fp

    # Any other obsm
    if export_other_obsm:
        for key in adata.obsm_keys():
            if key in other_obsm_exclude: continue
            arr = np.asarray(adata.obsm[key])
            if arr.ndim != 2 or arr.shape[0] != adata.n_obs: continue
            fp = os.path.join(output_dir, f"{key}.csv")
            pd.DataFrame(arr, index=adata.obs_names,
                         columns=[f"{key}_{i+1}" for i in range(arr.shape[1])]).to_csv(fp)
            written[f"obsm:{key}"] = fp

    if verbose:
        print(f"Export complete: {len(written)} files written to {output_dir}")
    return written


In [39]:
output_dir = OUTPUTS["quint_export_dir"]  # CSV export handed to the QUINT (R) step
written_files = export_anndata_for_seurat(adata_full, output_dir)


Export complete: 6 files written to C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\csvs\fmt


In [26]:
quint_df.value_counts()

quint_region                                            
Cerebral Nuclei, Striatum                                   56633
Fiber tract                                                 44661
Cerebellar cortex                                           43751
Olfactory                                                   37892
Cortex, Somatamotor areas                                   31112
Cortex, Somatasensory areas                                 23664
Hippocampus                                                 22069
Midbrain, sensory related                                   18410
Cortex, Visual areas                                        16474
Thalamus, polymodal association cortex related              14687
Hypothalamus                                                13057
Cerebral Nuclei, Pallidum                                   11922
medulla                                                     11425
Midbrain, motor related                                     11368
Cortex, Retrospleni

In [48]:

quint_df = pd.read_csv(INPUTS["quint_labels_csv"], index_col=0)

# See what you're aligning against (current adata vs. the CSV from R)
print("adata index :", list(adata_full.obs_names[:3]))
print("quint index :", list(quint_df.index[:3]))

# adata_full lost one trailing '-<n>' relative to the export R read back
# (sc.concat(index_unique='-') adds one each run). Shave the last '-<digits>'.
quint_df.index = quint_df.index.astype(str).str.replace(r'(-\d+){2}$', '', regex=True)

# verify BEFORE assigning — this is the silent failure you just hit
assert quint_df.index.is_unique, "stripping collapsed distinct cells — do not assign"
overlap = quint_df.index.intersection(adata_full.obs_names)
print(f"index match: {len(overlap)} / {adata_full.n_obs} cells")

adata_full.obs['quint_region'] = quint_df['quint_region']
print("NaN quint_region after assign:", int(adata_full.obs['quint_region'].isna().sum()))

adata index : ['1_1-2-0', '2_1-2-0', '3_1-2-0']
quint index : ['1_1-2-0-0-0', '2_1-2-0-0-0', '3_1-2-0-0-0']
index match: 446128 / 446128 cells
NaN quint_region after assign: 0


In [55]:
adata_full_.obs['quint_region'].value_counts()

quint_region
Cerebral Nuclei, Striatum                                   56633
Fiber tract                                                 44661
Cerebellar cortex                                           43751
Olfactory                                                   37892
Cortex, Somatamotor areas                                   31112
Cortex, Somatasensory areas                                 23664
Hippocampus                                                 22069
Midbrain, sensory related                                   18410
Cortex, Visual areas                                        16474
Thalamus, polymodal association cortex related              14687
Hypothalamus                                                13057
Cerebral Nuclei, Pallidum                                   11922
medulla                                                     11425
Midbrain, motor related                                     11368
Cortex, Retrosplenial area                                  104

In [49]:
adata_full_=adata_full.copy()

In [62]:
adata_full=adata_full_.copy()

In [67]:

adata_full = ezy.fix_stray_pixels(adata_full, region_col="quint_region", slide_col="slide_ID", reassign_background=True)

Keeping 1459 '0,0,0' background cells for reassignment
Stray (RGB) entries to reassign: 1459 / 446128
quint_region
0,0,0    1459
Name: count, dtype: int64
Using coordinates from obsm['spatial_fov']
Slide 120Land323DY: reassigned 740 stray cells
Slide 217Rand144_2L: reassigned 157 stray cells
Slide 367Rand215L: reassigned 329 stray cells
Slide 368R: reassigned 233 stray cells

=== quint_region after fix ===
quint_region
Cerebral Nuclei, Striatum                                   56653
Fiber tract                                                 44833
Cerebellar cortex                                           44172
Olfactory                                                   37912
Cortex, Somatamotor areas                                   31201
Cortex, Somatasensory areas                                 23709
Hippocampus                                                 22069
Midbrain, sensory related                                   18417
Cortex, Visual areas                             

In [68]:
adata_full.obs['quint_region'].value_counts()

quint_region
Cerebral Nuclei, Striatum                                   56653
Fiber tract                                                 44833
Cerebellar cortex                                           44172
Olfactory                                                   37912
Cortex, Somatamotor areas                                   31201
Cortex, Somatasensory areas                                 23709
Hippocampus                                                 22069
Midbrain, sensory related                                   18417
Cortex, Visual areas                                        16555
Thalamus, polymodal association cortex related              14687
Hypothalamus                                                13057
Cerebral Nuclei, Pallidum                                   11922
Midbrain, motor related                                     11465
medulla                                                     11426
Cortex, Retrosplenial area                                  105

In [5]:
r'''Verify Validity of QUINT labels'''


for slide in adata_full.obs['slide_ID'].unique():
    # Create subset for the current slide
    s_df = adata_full[adata_full.obs['slide_ID'] == slide].copy()

    print(f"Processing slide: {slide}")
    
    # --- PREPARE COLORS  ---
    color_by='cell_type' #This can be changed to any cellwise-data useful for visualization.
    unique_clusters = s_df.obs[color_by].unique()
    cmap = plt.get_cmap('tab20') 
    colors_list = [cmap(i) for i in np.linspace(0, 1, len(unique_clusters))]
    
    cluster_color_dict = dict(zip(unique_clusters, colors_list))
    assigned_colors = s_df.obs[color_by].astype(object).map(cluster_color_dict).tolist()

    # --- INITIALIZE VIEWER & ADD POINTS ---
    viewer = napari.Viewer()

    # Base cells
    viewer.add_points(
        s_df.obsm['spatial_fov'], 
        size=100,
        border_width=0.0,
        face_color=assigned_colors,
        name=slide
    )
    

Processing slide: 120Land323DY
Processing slide: 217Rand144_2L
Processing slide: 367Rand215L
Processing slide: 368R


In [70]:
# Create ct_simple column: Extract simplified cell types from region-specific names
# This reduces cell types like "Astrocytes.cortex.hippocampus" to just "Astrocytes"

# Get all unique cell types
unique_cell_types = adata_full.obs['cell_type'].unique()
print(f"Found {len(unique_cell_types)} unique cell types")

# Define multi-part patterns that should be kept together
# These patterns will be preserved as-is (e.g., "Excitatory.neurons.layer.1" -> "Excitatory.neurons")
multi_part_patterns = [
    "Excitatory.neurons",
    "Inhibitory.neurons", 
    "Interneuron",
    "Interneurons",
    "Astrocytes"
]

# Define cell types that should remain unchanged (no simplification)
unchanged_types = [
    "Mature.oligodendrocytes",
    "Myelin.forming.oligodendrocytes",
    "Committed.oligodendrocytes",
    "Granule.neurons",
    "Oligodendrocyte.precursor.cells",
    "Vascular.leptomeningeal.cells",
    "Vascular.smooth.muscle.cells",
    "Astrocytes.Bergmann.glia",
    "Purkinje.cells",
    "Radial.glia",
    "Ependymal.cells",
    "Neurogliaform.cells",
    "T.cell",
    "Serotonergic.neurons",
    "Newly.formed.oligodendrocytes",
    "CCK.interneurons",
    "Dopaminergic.neurons",
    "Peptidergic.neurons",
    "Olfactory.ensheathing.cells",
    "Cajal.Retzius.cells"
]

# Define specific mappings for special cases
special_mappings = {
    # --- FIX: Ensure all Interneuron variants map to the PLURAL "Interneurons" ---
    "Interneuron": "Interneurons",             
    "Inhibitory.interneurons": "Interneurons", 
    "Interneuron.selective.interneurons": "Interneurons",
    
    # Original mappings
    "D1.medium.spiny.neurons": "Spiny.neurons",
    "D2.medium.spiny.neurons": "Spiny.neurons",
    "Cholinergic.neurons.habenula": "Cholinergic.neurons",
    "Vascular.endothelial.cells": "Endothelial.cells",
    "Telencephalon.inhibitory.neurons": "Inhibitory.neurons",
    "Choroid.plexus.epithelial.cells": "Epithelial.cells",
    "Olfactory.bulb.inhibitory.neurons": "Inhibitory.neurons",
    "Hindbrain.inhibitory.neurons": "Inhibitory.neurons",
    "Hindbrain.excitatory.neurons": "Excitatory.neurons",

}

# Simplify these by stripping suffixes (Prefix -> General)
multi_part_patterns = [
    "Excitatory.neurons",
    "Inhibitory.neurons", 
    "Interneurons",
    "Astrocytes"
    # Note: "Interneuron" (singular) removed from patterns to avoid partial matches returning singular
]


# ------------------
def extract_simple_cell_type(cell_type):
    if pd.isna(cell_type):
        return cell_type
    
    ct_str = str(cell_type)
    
    # Unchanged (Catch specific complex names first)
    if ct_str in unchanged_types:
        return ct_str
    
    # Special Mappings (Fix specific overrides BEFORE pattern matching)
    if ct_str in special_mappings:
        return special_mappings[ct_str]
    
    # Pattern Matching (Prefix)
    for pattern in multi_part_patterns:
        # Check if it IS the pattern or STARTS with pattern + dot
        if ct_str == pattern or ct_str.startswith(pattern + "."):
            return pattern

    # Catch-all for singular "Interneuron" pattern misses
    if ct_str.startswith("Interneuron."):
        return "Interneurons"
            
    # Fallback
    return ct_str

# Apply Optimization
# ---------------------
print("\nGenerating mapping dictionary...")

# Get unique types and create a lookup dictionary
unique_types = adata_full.obs['cell_type'].unique()
type_mapping_dict = {ct: extract_simple_cell_type(ct) for ct in unique_types}

# Map the dictionary to the column
adata_full.obs['ct_simple'] = adata_full.obs['cell_type'].map(type_mapping_dict)

# Validation
# -------------
print("\n✓ ct_simple column created!")
print(f"Original unique types: {len(unique_types)}")
print(f"New unique types:      {adata_full.obs['ct_simple'].nunique()}")

print("\nValue counts for ct_simple:")
print(adata_full.obs['ct_simple'].value_counts())

Found 55 unique cell types

Generating mapping dictionary...

✓ ct_simple column created!
Original unique types: 55
New unique types:      34

Value counts for ct_simple:
ct_simple
Excitatory.neurons                 120220
Inhibitory.neurons                  53615
Astrocytes                          40325
Mature.oligodendrocytes             36493
Spiny.neurons                       29217
Myelin.forming.oligodendrocytes     25884
Committed.oligodendrocytes          23201
Endothelial.cells                   15483
Radial.glia                         13925
Pericytes                           12908
Interneurons                        12443
Hypendymal                          10330
Perivascular.macrophages             7713
Vascular.smooth.muscle.cells         6785
Granule.neurons                      5185
Oligodendrocyte.precursor.cells      4772
Astrocytes.Bergmann.glia             4583
Microglia                            4382
Vascular.leptomeningeal.cells        4327
Epithelial.cells     

In [72]:
adata_full.obs['napari_region'].value_counts()

napari_region
Unassigned     218650
Cortex          99954
Cerebellum      44777
Striatum        39215
Hippocampus     21167
Name: count, dtype: int64

In [ ]:
adata_full

In [1]:
adata_full=sc.read_h5ad(OUTPUTS["adata_h5ad"])


NameError: name 'sc' is not defined

In [71]:
# ── Save the fully-annotated cohort ──────────────────────────────────────────
# Written to config's adata_h5ad and loaded by Composition_Engineering.ipynb,
# which applies the cohort restriction, tags the composition-engineering
# scenarios, and exports the DE input. (Composition-engineering + DE export were
# moved out of this notebook into that separate pipeline step.)

adata_full=adata_full[adata_full.obs['napari_region']!='Olfactory']
adata_full.write_h5ad(OUTPUTS["adata_h5ad"])
print(f"Saved annotated cohort -> {OUTPUTS['adata_h5ad']}")


c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.

Saved annotated cohort -> C:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\Final_Data\fmt\adata_full.h5ad


In [ ]:
#create bar graphs
import matplotlib.ticker as ticker
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
stroke_color = (249/255,100/255,149/255)
healthy_color=(85/255,160/255,251/255)
cntrl_color=(184/255,86/255,215/255)



strata_col='Treatment'
strata_options=['Sham','Treatment']
def plot_celltype_counts_per_FMT(adata, top_n):

    ds=adata
    counts = ds.obs.groupby([strata_col, "ct_simple"]).size()
    top_clusters = (
        counts
        .groupby(level="ct_simple")
        .sum()
        .nlargest(top_n)
        .index
    )
    df = counts.unstack(level=strata_col).loc[top_clusters].fillna(0)
    df = df[strata_options]
    print(df.sum(axis=0))
    df_percent = df.div(df.sum(axis=0), axis=1) * 100
    print(df_percent)

    ax = df_percent.plot(
        kind="bar",
        color={strata_options[0]: healthy_color, strata_options[1]: stroke_color},
        figsize=(16, 12),
        width=0.8,
    )

    ax.legend( fontsize=28, title_fontsize=30, loc="upper right")
    ax.set_ylabel("Percentage of Cell Type", fontsize=30)
    print(f"Cell Cluster Abundance: Stroke-FMT vs. Healthy-FMT")
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=20)

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

plot_celltype_counts_per_FMT(adata_full, top_n=16)
